In [ ]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())
else:
    print("No GPU detected")

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

In [ ]:
dataset_path = r"C:\Users\Shivam\Downloads\archive\Driver Drowsiness Dataset (DDD)"

print(os.listdir(dataset_path))

In [ ]:
img_size = 224
batch_size = 32

transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

In [ ]:
dataset = datasets.ImageFolder(root=dataset_path, transform=transform)

print("Classes:", dataset.classes)
print("Class indices:", dataset.class_to_idx)

In [ ]:
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

In [ ]:
images, labels = next(iter(train_loader))

plt.figure(figsize=(10,6))
for i in range(6):
    plt.subplot(2,3,i+1)
    img = images[i].permute(1,2,0)
    plt.imshow((img * 0.5 + 0.5))  # unnormalize
    plt.title(dataset.classes[labels[i]])
    plt.axis("off")

plt.show()

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, 3),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 26 * 26, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

model = CNN().to(device)
print(model)

In [ ]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        predicted = (outputs > 0.5).float()
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Loss: {running_loss/len(train_loader):.4f} "
          f"Accuracy: {100*correct/total:.2f}%")

In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        outputs = model(images)
        predicted = (outputs > 0.5).float()

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Validation Accuracy:", 100 * correct / total)

In [ ]:
import torch
from torchvision import transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

model.eval()

In [ ]:
# import cv2
# from PIL import Image

# cap = cv2.VideoCapture(0)

# print("Press SPACE to capture and predict")
# print("Press ESC to exit")

# while True:
#     ret, frame = cap.read()
#     cv2.imshow("Driver Drowsiness Test", frame)

#     key = cv2.waitKey(1)

#     if key == 27:  # ESC
#         break

#     if key == 32:  # SPACE
#         image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#         image = Image.fromarray(image)

#         image_tensor = transform(image).unsqueeze(0).to(device)

#         with torch.no_grad():
#             output = model(image_tensor)
#             prediction = (output > 0.5).float()

#         label = "Non-Drowsy" if prediction.item() == 1 else "Drowsy"
#         confidence = output.item()

#         print("Prediction:", label)
#         print("Confidence:", confidence)

#         break

# cap.release()
# cv2.destroyAllWindows()

In [ ]:
torch.save(model.state_dict(), "driver_drowsiness_model.pth")

In [ ]:
import cv2
from PIL import Image

cap = cv2.VideoCapture(0)

print("Press SPACE to capture and predict")
print("Press ESC to exit")

while True:
    ret, frame = cap.read()
    cv2.imshow("Driver Drowsiness Test", frame)

    key = cv2.waitKey(1)

    if key == 27:  # ESC
        break

    if key == 32:  # SPACE
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image = Image.fromarray(image)

        image_tensor = transform(image).unsqueeze(0).to(device)

        with torch.no_grad():
            output = model(image_tensor)
            prediction = (output > 0.5).float()

        label = "Non-Drowsy" if prediction.item() == 1 else "Drowsy"
        confidence = output.item()

        print("Prediction:", label)
        print("Confidence:", confidence)                                     

        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
# Save only weights (you can load into same architecture later)
torch.save(model.state_dict(), "driver_drowsiness_model.pth")

# Optional: Save full model (easier to load later)
torch.save(model, "full_driver_drowsiness_model.pth")

In [ ]:
from IPython.display import FileLink

# Link to download full model
FileLink("full_driver_drowsiness_model.pth")

In [ ]:
pip install gdown